# S0.6R · 出境机制补「关键信息基础设施运营者」前提，训练人群与试点人群分开计数

**问题**：S0.5–S0.7 的判定矩阵按《促进和规范数据跨境流动规定》第五、七、八条推出：形态 A 在可验证规模下须申报数据出境安全评估，
形态 B 豁免。2026-10-25 复核（DR-MX-002 · D-3）发现两处前提没有进入推演：

1. 第五条第（四）项、第七条第（二）项与第八条只适用于**关键信息基础设施运营者以外**的数据处理者；
   关键信息基础设施运营者向境外提供个人信息，依第七条第（一）项一律申报。
2. 形态 A 中承载敏感个人信息的 CBA-005 梯度只对**训练人群**产生；N_eff ≥ 30,000 是**随机对照试点人群**。

本 notebook 把这两处补进判定矩阵。逻辑在 `modules/m0_compliance/components/export_mechanism.py`，
参数在 `modules/m0_compliance/configs/s0_6r_ciio_precondition.yaml`。

> ⚠️ 技术侧自评，未经法务复核（W-001），不构成法律意见。是否为关键信息基础设施运营者由保护工作部门认定并通知运营者，技术侧无法判定，故两种取值都列出。

In [1]:
import hashlib
import pathlib
import subprocess
import sys

import pandas as pd
import yaml

HASH_PREFIX_LEN = 12

REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

CONFIG_PATH = REPO_ROOT / "modules/m0_compliance/configs/s0_6r_ciio_precondition.yaml"
cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
config_hash = hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest()[:HASH_PREFIX_LEN]
git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True,
                                  cwd=REPO_ROOT).strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                       cwd=REPO_ROOT).stdout.strip() != ""
print("config:", CONFIG_PATH.relative_to(REPO_ROOT), "sha256:" + config_hash)
print("seed:", cfg["seed"])
print("git:", git_sha + ("（工作区含本步未提交改动）" if dirty else ""))
print("step:", cfg["step_id"], "|", cfg["purpose"])

config: modules/m0_compliance/configs/s0_6r_ciio_precondition.yaml sha256:158a3e6b7c82
seed: 42
git: ffea90e（工作区含本步未提交改动）
step: S0.6R | 在 S0.6 判定矩阵上增加「是否为关键信息基础设施运营者」与「是否适用大湾区标准合同」两个维度，并对形态 A 分开计训练人群与试点人群


In [2]:
from modules.m0_compliance.components.export_mechanism import required_mechanism

RESULTS = REPO_ROOT / "modules/m0_compliance/results"
thresholds = cfg["thresholds"]

records = []
for ciio in cfg["ciio_options"]:
    for gba in cfg["gba_options"]:
        for form in cfg["l3_forms"]:
            for n in cfg["scale_points"]:
                count = n if form["cn_to_hk_assets"] else 0
                records.append({
                    "关键信息基础设施运营者": "是" if ciio else "否",
                    "适用大湾区标准合同": "是" if gba else "否",
                    "L3 形态": form["id"],
                    "涉及人数": n,
                    "所需机制": required_mechanism(count, form["cn_to_hk_sensitive"], thresholds,
                                                gba_eligible=gba, is_ciio=ciio),
                })
matrix = pd.DataFrame(records)
matrix.to_csv(RESULTS / "export_mechanism_matrix_ciio.csv", index=False)

pivot = matrix.pivot_table(index=["关键信息基础设施运营者", "适用大湾区标准合同", "L3 形态"],
                           columns="涉及人数", values="所需机制", aggfunc="first")
print(pivot.to_string())

涉及人数                                 5000             30000            150000           1200000
关键信息基础设施运营者 适用大湾区标准合同 L3 形态                                                                    
否           否         形态A      标准合同或个人信息保护认证         数据出境安全评估         数据出境安全评估         数据出境安全评估
                      形态B     豁免（无需申报/订立/认证）   豁免（无需申报/订立/认证）    标准合同或个人信息保护认证         数据出境安全评估
                      形态C    不适用（该方向不提供个人信息）  不适用（该方向不提供个人信息）  不适用（该方向不提供个人信息）  不适用（该方向不提供个人信息）
            是         形态A     大湾区标准合同（免数量门槛）   大湾区标准合同（免数量门槛）   大湾区标准合同（免数量门槛）   大湾区标准合同（免数量门槛）
                      形态B     大湾区标准合同（免数量门槛）   大湾区标准合同（免数量门槛）   大湾区标准合同（免数量门槛）   大湾区标准合同（免数量门槛）
                      形态C    不适用（该方向不提供个人信息）  不适用（该方向不提供个人信息）  不适用（该方向不提供个人信息）  不适用（该方向不提供个人信息）
是           否         形态A           数据出境安全评估         数据出境安全评估         数据出境安全评估         数据出境安全评估
                      形态B           数据出境安全评估         数据出境安全评估         数据出境安全评估         数据出境安全评估
                      形态C    不适用（该方向不提供个

**读法**：

- 「关键信息基础设施运营者 = 否、适用大湾区标准合同 = 否」四行与 S0.6 已发布的 `export_mechanism_matrix.csv` 逐格一致（单元测试核对）。
- 一旦是关键信息基础设施运营者，形态 A、形态 B 在所有人数档上都落到**数据出境安全评估**，形态 B 的豁免不再成立；只有形态 C 不触发。
- 大湾区标准合同对关键信息基础设施运营者是否适用，实施指引没有规定；本表按 W-001 最严解释视为不可用。

In [3]:
from modules.m0_compliance.components.export_mechanism import strictest

split = cfg["form_a_split"]
pilot = split["pilot_population"]
rows = []
for ciio in cfg["ciio_options"]:
    for train in split["training_cohorts"]:
        identifiers = train + pilot if not split["identifier_population_overlap"] else max(train, pilot)
        gradient_rule = required_mechanism(train, True, thresholds, is_ciio=ciio)
        identifier_rule = required_mechanism(identifiers, False, thresholds, is_ciio=ciio)
        rows.append({
            "关键信息基础设施运营者": "是" if ciio else "否",
            "训练人群（梯度，敏感）": train,
            "标识符涉及人数（非敏感）": identifiers,
            "梯度所需机制": gradient_rule,
            "标识符所需机制": identifier_rule,
            "整体所需机制": strictest([gradient_rule, identifier_rule]),
            "S0.6 原口径（按试点人群计梯度）": required_mechanism(pilot, True, thresholds, is_ciio=ciio),
        })
split_table = pd.DataFrame(rows)
split_table.to_csv(RESULTS / "form_a_training_vs_pilot.csv", index=False)
print("试点人群（S0.1 N_eff）：", pilot)
print(split_table.to_string(index=False))

试点人群（S0.1 N_eff）： 30000
关键信息基础设施运营者  训练人群（梯度，敏感）  标识符涉及人数（非敏感）        梯度所需机制        标识符所需机制        整体所需机制 S0.6 原口径（按试点人群计梯度）
          否         5000         35000 标准合同或个人信息保护认证 豁免（无需申报/订立/认证） 标准合同或个人信息保护认证           数据出境安全评估
          否         9000         39000 标准合同或个人信息保护认证 豁免（无需申报/订立/认证） 标准合同或个人信息保护认证           数据出境安全评估
          否        30000         60000      数据出境安全评估 豁免（无需申报/订立/认证）      数据出境安全评估           数据出境安全评估
          是         5000         35000      数据出境安全评估       数据出境安全评估      数据出境安全评估           数据出境安全评估
          是         9000         39000      数据出境安全评估       数据出境安全评估      数据出境安全评估           数据出境安全评估
          是        30000         60000      数据出境安全评估       数据出境安全评估      数据出境安全评估           数据出境安全评估


## 结论

1. **是否为关键信息基础设施运营者，决定了形态 A/B 的区分在合规路径上还有没有意义。**
   如果是，形态 A、形态 B 都须申报数据出境安全评估，只有形态 C（不向香港侧提供任何个人信息）不触发。
   如果不是，S0.5–S0.7 的原结论不变。这个问题须由需求方回答，排在内地侧签约主体注册地之前。
2. **「可验证性与轻量出境路径不可兼得」取决于训练人群规模，而不是试点人群规模。**
   非关键信息基础设施运营者前提下，训练人群不满 1 万人时，形态 A 的整体机制是个人信息出境标准合同或个人信息保护认证；
   S0.6 按试点人群 30,000 人计梯度，得出的是数据出境安全评估。
   这一区分成立的前提是试点期 cn→hk 只有标识符流动，**须逐协议核对**。
3. 训练人群压到 1 万人以下，会受 S0.1P 的正例下界约束：训练集太小，离线比较也检不出差异。
   两者能否同时满足，取决于需求方给出的真实共同客户数与转化数。

> ⚠️ 本步不改 S0.5–S0.7 的已发布结果文件；新矩阵另存为 `export_mechanism_matrix_ciio.csv` 与 `form_a_training_vs_pilot.csv`。
> 全部判断为技术侧自评，未经法务复核（W-001）。